<a href="https://colab.research.google.com/github/janithcyapa/DHCA-Framework/blob/main/2.Thermal_Zone_Modeling_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thermal_Zone_Modeling_Verification

### Setup ENV

In [1]:
!pip uninstall -y energy-plus-utility

In [2]:
!pip install -q "energy-plus-utility @ git+https://github.com/janithcyapa/energy-plus-utility.git@main"
import importlib.metadata
ver = importlib.metadata.version("energy-plus-utility")
print(f"\n✅ Installed 'energy-plus-utility' version: {ver}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

✅ Installed 'energy-plus-utility' version: 0.2.2+1


In [ ]:
from eplus import prepare_colab_eplus
prepare_colab_eplus()

## Setup Model

In [10]:
from eplus.core import EPlusUtil
import types
import os
OUT_DIR = "/simulation/eplus_out"
url_idf = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/1ZoneUncontrolled_win_2.idf"
url_epw = "https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/System%20Models/Weather%20Files/LKA_Colombo-Katunayake.434500_SWERA.epw"

# Initialize Utility
sim = EPlusUtil(verbose=3, out_dir=OUT_DIR)
# Reset state before setting model
sim.reset_state()
# Delete previous output directory
sim.delete_out_dir()
# Clear previous outputs
sim.clear_eplus_outputs(patterns="eplusout.*")
# Set the Model for Simulation
sim.set_model_from_url(url_idf, url_epw)
sim.ensure_output_sqlite()



Initialized StateMixin
Initialized EnergyPlus State.
Initialized IDFMixin
Initialized LoggingMixin
Initialized SimulationMixin
Initialized UtilsMixin
Initialized HandlersMixin
Initialized SQLMixin
Initialized ControlMixin
Initialized OccupancyMixin
Initialized ZoneObserverMixin
EnergyPlus state has been reset.
Deleted output directory: /simulation/eplus_out
EnergyPlus state has been reset.
Model set: IDF='/simulation/eplus_out/1ZoneUncontrolled_win_2.idf', EPW='/simulation/eplus_out/LKA_Colombo-Katunayake.434500_SWERA.epw', OUT_DIR='/simulation/eplus_out'
EnergyPlus state has been reset.


'/simulation/eplus_out/1ZoneUncontrolled_win_2__sqlite.idf'

In [11]:
# sim.patch_idf_entry(
#     object_type="Construction:WindowDataFile",
#     object_name="DoubleClear",
#     old_value=r"..\datasets\Window5DataFile.dat",
#     new_value="/home/jazz/EnergyPlus-25-1-0/DataSets/Window5DataFile.dat"
# )
sim.patch_idf_entry(
    object_type="SimulationControl",
    object_name="No",
    old_value="No,                      !- Run Simulation for Weather File",
    new_value="Yes,                     !- Run Simulation for Weather File"
)

Patched 'No' (SimulationControl): Replaced 'No,                      !- Run Simulation for Weather File' -> 'Yes,                     !- Run Simulation for Weather File'


True

In [12]:
sim.run_dry_run(include_ems_edd=False,reset=True,design_day=False)
sim.run_design_day()

EnergyPlus state has been reset.
Deleted output file: /simulation/eplus_out/eplusout.err
Deleted output file: /simulation/eplus_out/eplusout.audit
EnergyPlus state has been reset.


1

## Setup Simulator

In [16]:


# Request the variables to construct State Vector (x_i)
specs = [
    {"name": "Zone Mean Air Temperature", "key": "*"},       # T_in
    # {"name": "Zone Mean Air Humidity Ratio", "key": "*"},    # W_in
    # {"name": "Zone Air CO2 Concentration", "key": "*"},      # C_in
    # {"name": "Site Outdoor Air Drybulb Temperature", "key": "Environment"}, # T_out
]
sim.ensure_output_variables(specs, activate=True)


def dr_supervisor_logic(self, state):
    """
    Supervisor logic that extracts date and time for synchronous data logging.
    """
    if not self.exchange.api_data_fully_ready(state):
        return

    # 1. Get current simulation time details
    day = self.exchange.day_of_year(state)
    time_now = self.exchange.time_of_day(state) # Hours (0.0 to 24.0)

    # Optional: Convert decimal hours to HH:MM for printing
    hours = int(time_now)
    minutes = int((time_now - hours) * 60)

    # 2. Extract your State Vector (x_i) via handles
    # Tip: Use the variable names from your specs list
    t_in_handle = self.exchange.get_variable_handle(state, "Zone Mean Air Temperature", "ZN001:ROOM001")
    t_in = self.exchange.get_variable_value(state, t_in_handle)

    print(f"[SUPERVISOR] Day: {day} | Time: {hours:02d}:{minutes:02d}", flush=True)

    # You can now append (day, time_now, t_in) to a list for manual plotting

sim.my_supervisor = types.MethodType(dr_supervisor_logic, sim)

registered = sim.register_handlers(
    "begin",               # Hook: Begin Timestep
    ["my_supervisor"]      # Method Name
)
print(f"Registered methods: {registered}")
current_list = sim.list_handlers("begin")
print(f"Handlers on 'begin' hook: {current_list}")
sim.set_simulation_params(
    start=(1, 1),           # January 1st
    end=(1, 7),             # January 7th
    start_day_of_week="Sunday"
)
print("Starting EnergyPlus Uncontrolled Simulation...")
sim.run_annual()
print("Simulation Complete!")



EnergyPlus state has been reset.
Registered methods: ['my_supervisor']
Handlers on 'begin' hook: ['my_supervisor']
EnergyPlus state has been reset.
Starting EnergyPlus Uncontrolled Simulation...
Deleted output file: /simulation/eplus_out/eplusout.err
EnergyPlus state has been reset.
Simulation Complete!


###

In [15]:
sql_path = OUT_DIR + "/eplusout.sql"
if os.path.exists(sql_path):

    # Plot the ground truth temperature to visually check it
    fig = sim.plot_sql_zone_variable(
        "Site Outdoor Air Drybulb Temperature",
        keys=["*"],
        reporting_freq=("TimeStep",),
        resample="1h",
        title="EnergyPlus Ground Truth: Uncontrolled Zone Temperature"
    )
    plt.show()
else:
    print("SQL output not found. Check if the simulation crashed.")

SQL output not found. Check if the simulation crashed.
